# Retail Inventory Analysis: Exploratory Data Analysis

## Objective
This notebook explores inventory and sales data to identify:
- Slow-moving items (low sales volume)
- Fast-selling products (high sales volume)
- Demand variability across stores and time periods

By analyzing these patterns, we can help retail chains optimize inventory management and reduce stockouts and overstocking.

## Section 1: Import Required Libraries

Import necessary libraries for data manipulation, analysis, and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

# Set visualization defaults
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

## Section 2: Load Raw Inventory Data

Read the CSV file from the data/raw/ folder and display basic information about the dataset.

In [ ]:
# Define data path
data_path = Path('..') / 'data' / 'raw' / 'sample_inventory_data.csv'

# Load data
inventory_data = pd.read_csv(data_path)

# Display basic information
print(f"Dataset shape: {inventory_data.shape} (rows, columns)")
print(f"\nFirst 10 rows of data:")
print(inventory_data.head(10))

print(f"\n\nColumn names and types:")
print(inventory_data.dtypes)

print(f"\n\nBasic statistics:")
print(inventory_data.describe())

## Section 3: Inspect Data Structure and Quality

Check data types, missing values, and identify any data quality issues.

In [ ]:
# Check for missing values
print("Missing values per column:")
print(inventory_data.isnull().sum())

# Check for duplicates
print(f"\n\nNumber of duplicate rows: {inventory_data.duplicated().sum()}")

# Check unique values in key columns
print(f"\n\nUnique values in key columns:")
print(f"Unique stores: {inventory_data['store_id'].nunique()}")
print(f"Unique products: {inventory_data['product_name'].nunique()}")
print(f"Product categories: {inventory_data['category'].unique()}")

# Check date range
print(f"\n\nDate range:")
print(f"Earliest date: {inventory_data['date'].min()}")
print(f"Latest date: {inventory_data['date'].max()}")

# Check for negative or zero values
print(f"\n\nData validation checks:")
print(f"Rows with negative units_sold: {(inventory_data['units_sold'] < 0).sum()}")
print(f"Rows with negative units_in_stock: {(inventory_data['units_in_stock'] < 0).sum()}")
print(f"Rows with zero or negative price: {(inventory_data['price'] <= 0).sum()}")

## Section 4: Explore Inventory Movement Patterns

Analyze sales trends over time and inventory levels to identify patterns.

In [ ]:
# Convert date to datetime
inventory_data['date'] = pd.to_datetime(inventory_data['date'])

# Calculate total sales by date
daily_sales = inventory_data.groupby('date')['units_sold'].sum()
print("Daily total sales:")
print(daily_sales)

# Calculate average inventory by date
daily_inventory = inventory_data.groupby('date')['units_in_stock'].mean()
print("\n\nDaily average inventory levels:")
print(daily_inventory)

# Calculate total revenue by date
inventory_data['revenue'] = inventory_data['units_sold'] * inventory_data['price']
daily_revenue = inventory_data.groupby('date')['revenue'].sum()
print("\n\nDaily total revenue:")
print(daily_revenue)

## Section 5: Analyze Product Performance Categories

Analyze which products drive revenue and which consume shelf space without sales.

In [ ]:
# Product performance analysis
product_performance = inventory_data.groupby('product_name').agg({
    'units_sold': ['sum', 'mean', 'count'],
    'revenue': 'sum',
    'units_in_stock': 'mean',
    'price': 'first'
}).round(2)

product_performance.columns = ['total_units_sold', 'avg_units_sold_per_day', 'sale_count', 
                                'total_revenue', 'avg_inventory', 'price']

# Sort by total revenue
product_performance = product_performance.sort_values('total_revenue', ascending=False)

print("Product Performance Summary:")
print(product_performance)

# Calculate inventory turnover ratio
product_performance['inventory_turnover'] = (
    product_performance['total_units_sold'] / product_performance['avg_inventory']
).round(2)

print("\n\nInventory Turnover Ratio (higher = faster moving):")
print(product_performance[['avg_units_sold_per_day', 'avg_inventory', 'inventory_turnover']].sort_values('inventory_turnover', ascending=False))

## Section 6: Examine Demand Variability Across Stores

Compare sales patterns and inventory levels between different store locations.

In [ ]:
# Store-level demand analysis
store_analysis = inventory_data.groupby('store_id').agg({
    'units_sold': ['sum', 'mean', 'std'],
    'revenue': 'sum',
    'units_in_stock': 'mean'
}).round(2)

store_analysis.columns = ['total_units_sold', 'avg_daily_sales', 'sales_std_dev', 
                          'total_revenue', 'avg_inventory']

print("Sales Pattern by Store:")
print(store_analysis)

# Calculate coefficient of variation (variability)
store_analysis['demand_variability'] = (
    store_analysis['sales_std_dev'] / store_analysis['avg_daily_sales']
).round(3)

print("\n\nDemand Variability by Store (higher = more variable demand):")
print(store_analysis[['avg_daily_sales', 'sales_std_dev', 'demand_variability']].sort_values('demand_variability', ascending=False))

## Section 7: Identify Slow-Moving and Fast-Selling Items

Use metrics like average sales to classify products as slow-movers or fast-sellers.

In [ ]:
# Calculate average daily sales per product
product_summary = inventory_data.groupby('product_name')['units_sold'].agg(['mean', 'median', 'std']).round(2)
product_summary.columns = ['avg_daily_sales', 'median_daily_sales', 'sales_std']

# Define thresholds
slow_moving_threshold = product_summary['avg_daily_sales'].quantile(0.33)
fast_selling_threshold = product_summary['avg_daily_sales'].quantile(0.67)

print(f"Product Sales Classification Thresholds:")
print(f"Slow-moving threshold (33rd percentile): {slow_moving_threshold:.1f} units/day")
print(f"Fast-selling threshold (67th percentile): {fast_selling_threshold:.1f} units/day")

# Classify products
slow_movers = product_summary[product_summary['avg_daily_sales'] < slow_moving_threshold].sort_values('avg_daily_sales')
fast_sellers = product_summary[product_summary['avg_daily_sales'] >= fast_selling_threshold].sort_values('avg_daily_sales', ascending=False)

print("\n\nSLOW-MOVING ITEMS (Low sales volume):")
print(slow_movers)

print("\n\nFAST-SELLING ITEMS (High sales volume):")
print(fast_sellers)

## Section 8: Visualize Key Findings

Create plots to illustrate inventory patterns and product performance across stores.

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Daily sales trend
axes[0, 0].plot(daily_sales.index, daily_sales.values, marker='o', linewidth=2)
axes[0, 0].set_title('Daily Total Sales Trend', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Units Sold')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Product performance (average daily sales)
product_summary['avg_daily_sales'].sort_values(ascending=True).plot(
    kind='barh', ax=axes[0, 1], color=['red' if x < slow_moving_threshold else 'green' for x in product_summary['avg_daily_sales'].sort_values().values]
)
axes[0, 1].set_title('Average Daily Sales by Product', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Units Sold')
axes[0, 1].axvline(slow_moving_threshold, color='orange', linestyle='--', label='Slow-moving threshold')
axes[0, 1].axvline(fast_selling_threshold, color='blue', linestyle='--', label='Fast-selling threshold')
axes[0, 1].legend()

# 3. Sales by store
store_sales = inventory_data.groupby('store_id')['units_sold'].sum()
axes[1, 0].bar(store_sales.index, store_sales.values, color='steelblue')
axes[1, 0].set_title('Total Sales by Store', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Store ID')
axes[1, 0].set_ylabel('Units Sold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. Inventory levels over time
for product in inventory_data['product_name'].unique():
    product_data = inventory_data[inventory_data['product_name'] == product].groupby('date')['units_in_stock'].mean()
    axes[1, 1].plot(product_data.index, product_data.values, marker='o', label=product)
axes[1, 1].set_title('Inventory Levels Over Time', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Average Units in Stock')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("✓ Visualizations created successfully")

## Section 9: Document Insights and Next Steps

### Key Findings Summary

Based on the exploratory data analysis, here are the main insights:

**1. Product Performance:**
- **Fast-Sellers:** Products with high average daily sales (> 67th percentile) generate significant revenue and should be prioritized
- **Slow-Movers:** Products with low sales volume consume shelf space and inventory capital with minimal returns

**2. Demand Variability:**
- Different stores show varying demand patterns and variability
- Understanding store-specific demand helps in targeted inventory allocation
- Stores with high demand variability face greater challenges in optimal stock planning

**3. Inventory Efficiency:**
- Inventory turnover ratios indicate how efficiently stock is being converted to sales
- High turnover = efficient inventory management; Low turnover = potential dead stock

### Recommendations

1. **Optimize Slow-Moving Items:** Consider clearance sales, repositioning, or discontinuation of items with consistently low sales
2. **Stock Planning:** Allocate more inventory to fast-sellers and adjust for store-specific demand patterns
3. **Inventory Targets:** Use historical variability to set appropriate safety stock levels
4. **Regular Monitoring:** Conduct periodic reviews to track performance changes and adjust strategies accordingly

### Next Steps

1. Run the data processing script (`load_and_clean_data.py`) to prepare data for detailed analysis
2. Execute the analysis script (`analyze_inventory.py`) to generate summary reports and metrics
3. Use findings to inform purchasing, pricing, and merchandising decisions
4. Monitor performance metrics over time to validate improvements